<a href="https://colab.research.google.com/github/Saphythrix/IIT-SUMMER-INTERNSHIP/blob/main/Copy_of_CIFAR_10_image_Gradient.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed(42)

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [ ]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5,0.5,0.5),
        std=(0.5,0.5,0.5)
    ),
    transforms.Lambda(lambda x:x.view(3,-1))
])

In [ ]:
train_dataset=torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)
test_dataset=torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [ ]:
train_loader=DataLoader(train_dataset,batch_size=512,shuffle=True,pin_memory=True,num_workers=4)
test_loader=DataLoader(test_dataset,batch_size=512,shuffle=False,pin_memory=True,num_workers=4)

In [ ]:
CLASSES = train_dataset.classes
C = len(CLASSES)
print(f"Classes: {CLASSES}")
print(f"Input shape per sample: {train_dataset[0][0].shape}")

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Input shape per sample: torch.Size([3, 1024])


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

#Bulding the model
class TemporalCNN1D(nn.Module):

  def __init__(self,num_classes=10,dropout_p=0.5):
      super().__init__()

      def conv_block(input_ch,output_ch,apply_drop=True):
          layers=[
              nn.Conv1d(input_ch,output_ch,kernel_size=3,padding=1),
              nn.BatchNorm1d(output_ch,affine=True),
              nn.LeakyReLU(negative_slope=0.3)
          ]
          if apply_drop:
            layers.append(nn.Dropout(p=dropout_p))
          return nn.Sequential(*layers)

      self.encoder=nn.Sequential(
          conv_block(3,32,apply_drop=True),
          conv_block(32,64,apply_drop=True),
          conv_block(64,64,apply_drop=True),
          conv_block(64,128,apply_drop=True),
          conv_block(128,128,apply_drop=True),
          conv_block(128,100,apply_drop=False)
      )

      self.pool=nn.AdaptiveAvgPool1d(1)

      self.classifer=nn.Linear(100,num_classes)

  def forward(self,x):
      features=self.encoder(x)
      pooling=self.pool(features)
      flatten=pooling.squeeze(-1)
      logit=self.classifer(flatten)
      return logit

  def get_features(self,x):
      features=self.encoder(x)
      pooled=self.pool(features)

      return pooled.squeeze(-1)

In [ ]:
learning_rate = 0.001
epochs = 25

In [ ]:
model=TemporalCNN1D(num_classes=10,dropout_p=0.5).to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=learning_rate)

In [ ]:
#training loop
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []} # Initialize history

for epoch in range(epochs):
  # Training
  model.train()
  total_train_loss = 0
  correct_train_predictions = 0
  total_train_samples = 0

  for inputs,labels in train_loader:
    inputs,labels=inputs.to(device),labels.to(device)

    # forward pass
    output=model(inputs)
    loss=criterion(output,labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_train_loss += loss.item() * inputs.size(0)
    correct_train_predictions    += (output.argmax(dim=1) == labels).sum().item() # Use output instead of logits
    total_train_samples      += inputs.size(0)

  # Calculate epoch-level metrics
  avg_train_loss = total_train_loss / total_train_samples
  train_accuracy = correct_train_predictions / total_train_samples
  history["train_loss"].append(avg_train_loss)
  history["train_acc"].append(train_accuracy)

  # Validation
  model.eval()
  total_val_loss = 0
  correct_val_predictions = 0
  total_val_samples = 0

  with torch.no_grad():
    for inputs, labels in test_loader: # Assuming test_loader is the validation loader
      inputs, labels = inputs.to(device), labels.to(device)
      output = model(inputs)
      loss = criterion(output, labels)

      total_val_loss += loss.item() * inputs.size(0)
      correct_val_predictions += (output.argmax(dim=1) == labels).sum().item()
      total_val_samples += inputs.size(0)

  avg_val_loss = total_val_loss / total_val_samples
  val_accuracy = correct_val_predictions / total_val_samples
  history["val_loss"].append(avg_val_loss)
  history["val_acc"].append(val_accuracy)

  print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.4f} | Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history["train_loss"], label="Train Loss")
ax1.plot(history["val_loss"],   label="Val Loss")
ax1.set_title("Loss over Epochs")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot([a*100 for a in history["train_acc"]], label="Train Acc")
ax2.plot([a*100 for a in history["val_acc"]],   label="Val Acc")
ax2.set_title("Accuracy over Epochs")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:

print(model)
last_layer=list(model.children())[-1]
print(last_layer)

 Reference Gradient (g_s)

In [ ]:
model.eval()
x_train_batch, y_train_batch = next(iter(train_loader))
print(x_train_batch.shape)
print(y_train_batch.shape)

ref_images=x_train_batch.to(device)
ref_labels=y_train_batch.to(device)

model.zero_grad()
#forward propagation
output=model(ref_images)

#loss calculation
loss=criterion(output,ref_labels)


loss.backward()

g_s=last_layer.weight.grad.detach().cpu().numpy().flatten()
print(g_s.shape)
print(g_s)
last_layer.weight.shape

Runtime Gradient (g_m)

In [ ]:
x_test_batch, y_test_batch = next(iter(test_loader))
x_test_batch=x_test_batch.to(device)
y_test_batch=y_test_batch.to(device)
model.eval()
model.zero_grad()

output=model(x_test_batch)
loss=criterion(output,y_test_batch)

loss.backward()

g_m=last_layer.weight.grad.detach().cpu().numpy().flatten()
print(g_m)
print(g_m.shape)

Gradient Angle

In [ ]:
def gradient_angle(g_s,g_m):
  dot_pro=np.dot(g_s,g_m)
  g_m_norm=np.linalg.norm(g_m)
  g_s_norm=np.linalg.norm(g_s)

  cosine=np.clip(dot_pro/(g_m_norm*g_s_norm),-1.0,1.0)
  angle=np.degrees(np.arccos(cosine))

  return angle

grad_angle=gradient_angle(g_s,g_m)
print(f"Gradient Angle: {grad_angle:.2f}°")

Gradient Angle: 26.42°
